# The Self-Pruning Neural Network
**Smooth Gradual Sparsity Edition**
This notebook implements a self-pruning neural network on CIFAR-10. It features a Delayed Sparsity (Warmup) schedule and gradient clipping to smoothly isolate and prune unnecessary connections without collapsing network performance.
---

In [ ]:
import math
import json
import logging
from typing import Tuple, List

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

def set_seed(seed: int = 42) -> None:
    '''Sets the seed for reproducibility.'''
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")

### 1. Define the Prunable Layer


In [ ]:
class PrunableLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias = nn.Parameter(torch.Tensor(out_features))
        self.gate_scores = nn.Parameter(torch.Tensor(out_features, in_features))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(self.bias, -bound, bound)
        nn.init.constant_(self.gate_scores, 1.0)

    def get_gates(self) -> torch.Tensor:
        return torch.sigmoid(self.gate_scores)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gates = self.get_gates()
        pruned_weight = self.weight * gates
        return F.linear(x, pruned_weight, self.bias)

### 2. Define the Neural Network Architecture & Sparsity Logics

In [ ]:
class SelfPruningNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = PrunableLinear(3072, 256)
        self.fc2 = PrunableLinear(256, 128)
        self.fc3 = PrunableLinear(128, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def get_sparsity_loss(self) -> torch.Tensor:
        l1_loss = torch.tensor(0.0, device=next(self.parameters()).device)
        for module in self.modules():
            if isinstance(module, PrunableLinear):
                l1_loss += torch.sum(module.get_gates())
        return l1_loss

    def get_sparsity_metrics(self, threshold: float = 1e-2) -> Tuple[float, np.ndarray]:
        total_weights = 0
        pruned_weights = 0
        all_gates = []
        
        for module in self.modules():
            if isinstance(module, PrunableLinear):
                gates = module.get_gates().detach()
                total_weights += gates.numel()
                pruned_weights += torch.sum(gates < threshold).item()
                all_gates.append(gates.cpu().numpy().flatten())
                
        sparsity_pct = (pruned_weights / total_weights) * 100.0 if total_weights > 0 else 0.0
        return sparsity_pct, np.concatenate(all_gates)

### 3. Training & Evaluation Pipeline (Warmup & Gradient Clipping)

In [ ]:
def train_model(target_lmbda: float, epochs: int = 12, warmup_epochs: int = 3, batch_size: int = 256):
    print(f"\n{'='*50}")
    print(f"Starting training with Target λ = {target_lmbda}")
    print(f"{'='*50}")
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    ])
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

    model = SelfPruningNet().to(device)
    classification_criterion = nn.CrossEntropyLoss()
    
    # Split LR to allow stable gradient convergence
    gate_params = [p for n, p in model.named_parameters() if 'gate_score' in n]
    weight_params = [p for n, p in model.named_parameters() if 'gate_score' not in n]
    optimizer = optim.Adam([
        {'params': weight_params, 'lr': 2e-3},
        {'params': gate_params, 'lr': 1.5e-2}
    ])

    for epoch in range(epochs):
        model.train()
        running_loss, running_cls_loss, running_sparse_loss = 0.0, 0.0, 0.0
        current_lmbda = target_lmbda * min(1.0, epoch / warmup_epochs) if warmup_epochs > 0 else target_lmbda
        
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            cls_loss = classification_criterion(outputs, labels)
            sparsity_loss = model.get_sparsity_loss()
            
            loss = cls_loss + current_lmbda * sparsity_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gate_params, max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()
            running_cls_loss += cls_loss.item()
            running_sparse_loss += sparsity_loss.item()
            
        print(
            f"Epoch {epoch+1:02d}/{epochs} [λ={current_lmbda:.5f}] - "
            f"Total Loss: {running_loss/len(trainloader):.4f} | "
            f"CE Loss: {running_cls_loss/len(trainloader):.4f} | "
            f"L1 Sparsity Loss: {running_sparse_loss/len(trainloader):.4f}"
        )

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_accuracy = 100 * correct / total
    sparsity_pct, all_gates = model.get_sparsity_metrics(threshold=1e-2)
    print(f"\n=> Accuracy: {test_accuracy:.2f}% | Sparsity (< 1e-2): {sparsity_pct:.2f}%\n")
    return test_accuracy, sparsity_pct, all_gates

### 4. Run Model Execution Experiment

In [ ]:
lambdas = [1e-5, 5e-5, 1e-4, 5e-4]
results = {}
acc_list, sparse_list = [], []
best_lambda, best_gates = None, None

for lmbda in lambdas:
    acc, spars, gates = train_model(lmbda, epochs=12, warmup_epochs=3)
    results[lmbda] = {"accuracy": acc, "sparsity": spars}
    acc_list.append(acc)
    sparse_list.append(spars)
    
    if spars > 50.0:
        best_lambda = lmbda
        best_gates = gates

print("\n--- Smooth Self-Pruning Tradeoff Results ---")
for lmbda in lambdas:
    print(f"Lambda: {lmbda:<8} | Accuracy: {results[lmbda]['accuracy']:>5.2f}% | Sparsity (< 1e-2): {results[lmbda]['sparsity']:>5.2f}%")

### 5. Tradeoff Visualization Line Graph

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))
lambda_labels = [f"{l}" for l in lambdas]

ax1.set_xlabel('Lambda Penalty')
ax1.set_ylabel('Accuracy (%)', color='tab:blue', fontweight='bold')
ax1.plot(lambda_labels, acc_list, marker='o', color='tab:blue', linewidth=2, label='Accuracy')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.set_ylabel('Sparsity (%)', color='tab:red', fontweight='bold')
ax2.plot(lambda_labels, sparse_list, marker='s', color='tab:red', linewidth=2, linestyle='dashed', label='Sparsity')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title("Gradual Sparsity vs Accuracy Tradeoff", fontweight='bold', fontsize=14)
fig.tight_layout()
plt.grid(alpha=0.3)
plt.show()

### 6. Final Mask Network State Histogram

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(best_gates, bins=50, alpha=0.75, color='indigo', edgecolor='black', label=f'Best Smooth Gates (λ={best_lambda})')
plt.title("Distribution of Final Gating Values Across Network", fontsize=14, fontweight='bold')
plt.xlabel("Gate Value (0.0 = completely pruned, 1.0 = fully active)", fontsize=12)
plt.ylabel("Parameter Frequency", fontsize=12)
plt.legend(loc='upper right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()